# 03 – Modelowanie i ewaluacja
**Cel:** Testowanie hipotez statystycznych oraz budowa i porównanie modeli predykcyjnych ceny samochodu.

**Pytania badawcze:**
1. Jakie czynniki mają największy wpływ na cenę?
2. Jak dobrze można przewidzieć cenę na podstawie parametrów pojazdu?
3. Czy Random Forest / XGBoost bije regresję liniową?

## 1. Import i wczytanie danych

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from xgboost import XGBRegressor
    USE_XGB = True
except ImportError:
    USE_XGB = False
    print('XGBoost niedostępny.')
from pathlib import Path
import os
PROJECT_ROOT = Path(os.getcwd())
# Jeśli CWD to notebooks/, cofnij się poziom wyżej
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH    = str(PROJECT_ROOT / 'data' / 'raw' / 'Car_Prices_Poland_Kaggle.csv')
PROC_PATH    = str(PROJECT_ROOT / 'data' / 'processed') + os.sep
FIGURES_PATH = str(PROJECT_ROOT / 'reports' / 'figures') + os.sep
os.makedirs(PROC_PATH, exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('FIGURES_PATH:', FIGURES_PATH)

In [ ]:
df = pd.read_csv(PROC_PATH + 'cars_clean.csv')
print(f'Wczytano: {df.shape}')
df.head(3)

## 2. Testy statystyczne
### 2.1 ANOVA – czy rodzaj paliwa różnicuje cenę?

In [ ]:
fuel_groups = [group['price'].values for _, group in df.groupby('fuel')]
f_stat, p_anova = stats.f_oneway(*fuel_groups)
print(f'ANOVA F-statystyka: {f_stat:.2f},  p-wartość: {p_anova:.4e}')
if p_anova < 0.05:
    print('→ Odrzucamy H0: istnieją statystycznie istotne różnice cen między grupami paliw (α=0.05)')
else:
    print('→ Brak podstaw do odrzucenia H0.')

### 2.2 Kruskal-Wallis

In [ ]:
h_stat, p_kw = stats.kruskal(*fuel_groups)
print(f'Kruskal-Wallis H: {h_stat:.2f},  p-wartość: {p_kw:.4e}')
if p_kw < 0.05:
    print('→ Różnice cen między grupami paliw są statystycznie istotne.')

### 2.3 Test t – diesel vs. benzyna

In [ ]:
diesel = df[df['fuel'] == 'diesel']['price']
petrol_keys = [k for k in df['fuel'].unique() if any(x in k for x in ['petrol','benzyna','gasoline','gas'])]
petrol_key = petrol_keys[0] if petrol_keys else None
if petrol_key:
    petrol = df[df['fuel'] == petrol_key]['price']
    t_stat, p_t = stats.ttest_ind(diesel, petrol, equal_var=False)
    print(f'Test t (diesel vs. {petrol_key}): t={t_stat:.3f}, p={p_t:.4e}')
    print(f'Mediana diesel: {diesel.median():,.0f} PLN')
    print(f'Mediana {petrol_key}: {petrol.median():,.0f} PLN')
else:
    print('Dostępne rodzaje paliwa:')
    print(df['fuel'].value_counts())

### 2.4 Korelacja Spearmana

In [ ]:
num_features = [c for c in ['year','mileage','vol_engine','car_age','mileage_per_year','mark_median_price'] if c in df.columns]
rows = []
for col in num_features:
    mask = df[col].notna()
    rho, p = stats.spearmanr(df.loc[mask, col], df.loc[mask, 'price'])
    rows.append({'Cecha': col, 'rho': round(rho, 4), 'p-wartość': f'{p:.2e}'})
sp_df = pd.DataFrame(rows).sort_values('rho', key=abs, ascending=False)
print(sp_df.to_string(index=False))

## 3. Przygotowanie danych do modelowania

In [ ]:
base_features = ['year','mileage','vol_engine','car_age','mileage_per_year','mark_median_price','is_electric_hybrid']
fuel_cols = [c for c in df.columns if c.startswith('fuel_')]
FEATURES = [c for c in base_features + fuel_cols if c in df.columns]
TARGET = 'log_price'
df_model = df[FEATURES + [TARGET, 'price']].dropna()
print(f'Cechy ({len(FEATURES)}): {FEATURES}')
print(f'Rozmiar zbioru: {df_model.shape}')
X = df_model[FEATURES]
y = df_model[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 4. Trening modeli

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = np.expm1(model.predict(X_te))
    true  = np.expm1(y_te)
    rmse  = np.sqrt(mean_squared_error(true, preds))
    mae   = mean_absolute_error(true, preds)
    r2    = r2_score(true, preds)
    cv_r2 = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()
    print(f'{name:30s}  R2={r2:.4f}  RMSE={rmse:,.0f}  MAE={mae:,.0f}  CV-R2={cv_r2:.4f}')
    return {'Model': name, 'RMSE': round(rmse, 0), 'MAE': round(mae, 0),
            'R2': round(r2, 4), 'CV R2 (5-fold)': round(cv_r2, 4),
            '_model': model, '_preds': preds}

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)

results = []
results.append(evaluate('Linear Regression', LinearRegression(), X_tr_sc, y_train, X_te_sc, y_test))
results.append(evaluate('Ridge (a=1)',        Ridge(alpha=1.0),   X_tr_sc, y_train, X_te_sc, y_test))
results.append(evaluate('Lasso (a=0.001)',    Lasso(alpha=0.001, max_iter=10000), X_tr_sc, y_train, X_te_sc, y_test))
results.append(evaluate('Random Forest',      RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), X_train, y_train, X_test, y_test))
if USE_XGB:
    results.append(evaluate('XGBoost', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, verbosity=0), X_train, y_train, X_test, y_test))

In [ ]:
metrics_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results])
metrics_df

## 5. Wizualizacja – porównanie modeli

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (metric, higher_better) in zip(axes, [('R2', True), ('RMSE', False), ('MAE', False)]):
    sdf = metrics_df.sort_values(metric, ascending=not higher_better).reset_index(drop=True)
    colors = ['darkgreen' if i == 0 else 'steelblue' for i in range(len(sdf))]
    ax.barh(sdf['Model'], sdf[metric], color=colors)
    ax.set_title(metric.replace('R2', 'R²'))
    ax.set_xlabel(metric.replace('R2', 'R²'))
plt.suptitle('Porównanie modeli predykcyjnych', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_PATH + '03_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Najlepszy model – residua i ważność cech

In [ ]:
best_idx   = metrics_df['R2'].idxmax()
best_res   = results[best_idx]
best_name  = best_res['Model']
best_preds = best_res['_preds']
print(f'Najlepszy model: {best_name}  (R2={best_res["R2"]})')
true_vals = np.expm1(y_test.values)
residuals = true_vals - best_preds

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(best_preds, true_vals, alpha=0.2, s=8, color='steelblue')
lims = [min(best_preds.min(), true_vals.min()), max(best_preds.max(), true_vals.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='idealna predykcja')
axes[0].set_title(f'{best_name}\nPrzewidywane vs. Rzeczywiste')
axes[0].set_xlabel('Przewidywana cena (PLN)')
axes[0].set_ylabel('Rzeczywista cena (PLN)')
axes[0].legend()
axes[1].scatter(best_preds, residuals, alpha=0.2, s=8, color='darkorange')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residua vs. Przewidywana cena')
axes[1].set_xlabel('Przewidywana cena (PLN)')
axes[1].set_ylabel('Residuum (PLN)')
plt.tight_layout()
plt.savefig(FIGURES_PATH + '03_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_model_obj = best_res['_model']
if hasattr(best_model_obj, 'feature_importances_'):
    imp = pd.Series(best_model_obj.feature_importances_, index=FEATURES).sort_values(ascending=False).head(15)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=imp.values, y=imp.index, palette='viridis')
    plt.title(f'Ważność cech – {best_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig(FIGURES_PATH + '03_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
elif hasattr(best_model_obj, 'coef_'):
    coefs = pd.Series(best_model_obj.coef_, index=FEATURES).abs().sort_values(ascending=False).head(15)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=coefs.values, y=coefs.index, palette='viridis')
    plt.title(f'Współczynniki (abs.) – {best_name}')
    plt.xlabel('|Coefficient|')
    plt.tight_layout()
    plt.savefig(FIGURES_PATH + '03_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Zapis wyników

In [ ]:
metrics_df.to_csv(PROC_PATH + 'model_results.csv', index=False)
print('Wyniki zapisane do:', PROC_PATH + 'model_results.csv')
metrics_df

## 8. Wnioski z modelowania

| Hipoteza | Wynik |
|---|---|
| H1: Rok produkcji i przebieg są najsilniejszymi predyktorami | ✅/❌ (sprawdź feature importance) |
| H2: Samochody elektryczne/hybrydowe są droższe | ✅/❌ (sprawdź test ANOVA) |
| H3: RF/XGBoost bije regresję liniową | ✅/❌ (sprawdź tabelę R²) |

> Uzupełnij tabelę po uruchomieniu notebooka.